# 04 — Explanation and advice · Crowdfund Compass (Group 11)
Goal: turn the final model into things a coach or founder can use: why a campaign scores as it does, six area scores, the single changes that are associated with better odds, and a verdict.

Workflow: 1) Setup → 2) Load model and data → 3) SHAP → 4) Area scores → 5) Chance range and verdict rule → 6) What to change → 7) Product types that do badly → 8) Explanation tree → 9) Examples → 10) Conclusions.

**Read this first:** everything here describes associations in past campaigns. A suggested change is "campaigns that did this were funded more often", never a promise.

Run notebook 03 first (it saves the model). *AI use: code written with Claude (Fable 5.2), 16.11.2026; reviewed by the team.*

## 1) Setup

In [1]:
import sys, time, json, warnings, os
warnings.filterwarnings('ignore')
!pip -q install vaderSentiment
import numpy as np, pandas as pd, joblib, xgboost as xgb
from sklearn.tree import DecisionTreeClassifier, export_text
from sklearn.metrics import roc_auc_score
pd.set_option('display.width', 250); pd.set_option('display.max_rows', 200); pd.set_option('display.max_colwidth', 90)

# from google.colab import drive; drive.mount('/content/drive')
project = 'drive/My Drive/crowdfund_compass'      # folder with features.py, modeling.py, advice.py
out = project + '/models'                         # written by notebook 03
sys.path.insert(0, project)
import features as F, modeling as M, advice as A

## 2) Load the model and the data
Training campaigns (2023–24) are the reference population; the 2025 test year is used to check everything.

In [2]:
ART=joblib.load(out+'/models_M.joblib'); FM=ART['features']
D=M.prepare(pd.read_parquet(out+'/features.parquet'))
TR=D[D.period.isin(['2023','2024'])&D.has_video.notna()].copy(); TE=D[D.period=='2025'].copy()
oof,(ts,),_=M.text_score(TR,[TE]); TR['text_score']=oof; TE['text_score']=ts
adv=A.Advisor(ART)

## 3) SHAP: what drives the score? (beyond class)
*Why:* a coach needs reasons, not only a number. *In plain words:* SHAP splits each campaign's score into one piece per feature: how much that feature pushed the score up or down compared with an average campaign. The pieces add up exactly to the model's score (checked below).

In [3]:
# ---- 1) SHAP: global importance
S=adv.shap(TE); imp=S.abs().mean().sort_values(ascending=False)
print('mean |SHAP| top 12 (log-odds):'); print(imp.head(12).round(3).to_string())
G=pd.Series({a:S[c].abs().sum(axis=1).mean() for a,c in A.AREAS.items()}); print('by area, share of total:'); print((G/G.sum()).round(3).sort_values(ascending=False).to_string())
(G/G.sum()).to_csv(out+'/shap_by_area.csv'); imp.to_csv(out+'/shap_by_feature.csv')
# additivity check: SHAP sums + base = model margin
c=ART['model'].get_booster().predict(xgb.DMatrix(TE[FM]),pred_contribs=True); raw=ART['model'].predict_proba(TE[FM])[:,1]
print('SHAP additivity: max |sum - logit(p)| =',float(np.abs(c.sum(axis=1)-np.log(raw/(1-raw))).max().round(5)))

mean |SHAP| top 12 (log-odds):
text_score              0.758
creator_prior_rate      0.587
cat_id                  0.369
log_goal_usd            0.337
prelaunch_activated     0.291
log_prep_days           0.245
creator_prior_funded    0.227
duration_days           0.180
has_video               0.138
parent_c                0.079
log_goal_per_day        0.072
goal_ratio              0.069
by area, share of total:
Market          0.342
Founder         0.228
Ambition        0.188
Presentation    0.116
Timing          0.087
Pitch           0.039


SHAP additivity: max |sum - logit(p)| = 2.9999999242136255e-05


**Interpretation:** what the product is (Market) and who launches it (Founder) account for more than half of the explanation. The areas a founder can change before launch (Ambition, Presentation, Timing, Pitch) account for the rest.

## 4) Area scores, 0–100
For each of the six areas we add up the SHAP pieces of its features and ask: what share of the reference campaigns had a lower sum? 50 = typical, 100 = this area helps more than in every reference campaign. Bands: 0–25 weak (red), 26–60 to improve (amber), 61–100 strong (green). The check below shows that campaigns with a weak area were funded less often in 2025.

In [4]:
# ---- 2) area scores: reference = training campaigns 2023-24
ref=adv.area_sums(TR); ART['area_reference']={k:np.sort(ref[k].values) for k in A.AREAS}; adv.ref=ART['area_reference']
AS=adv.area_scores(TE); TE=TE.join(AS.add_prefix('score_'))
for a in A.AREAS:
    b=pd.cut(TE['score_'+a],[-1,25,60,100],labels=['weak (0-25)','to improve (26-60)','strong (61-100)'])
    t=TE.groupby(b,observed=True).success.agg(n='size',funded='mean'); print(a, ' | '.join(f'{i}: n={int(r.n)}, funded {r.funded:.2f}' for i,r in t.iterrows()))

Ambition weak (0-25): n=6308, funded 0.38 | to improve (26-60): n=9104, funded 0.75 | strong (61-100): n=11121, funded 0.86
Timing weak (0-25): n=6390, funded 0.46 | to improve (26-60): n=8703, funded 0.77 | strong (61-100): n=11440, funded 0.80
Pitch weak (0-25): n=5536, funded 0.65 | to improve (26-60): n=8199, funded 0.66 | strong (61-100): n=12798, funded 0.76
Founder weak (0-25): n=6307, funded 0.41 | to improve (26-60): n=8705, funded 0.62 | strong (61-100): n=11521, funded 0.94
Presentation weak (0-25): n=6079, funded 0.55 | to improve (26-60): n=9442, funded 0.68 | strong (61-100): n=11012, funded 0.82
Market weak (0-25): n=5739, funded 0.29 | to improve (26-60): n=9018, funded 0.69 | strong (61-100): n=11776, funded 0.92


## 5) Chance range and the verdict rule
- **High end** of the range: calibrated chance in our sample. **Low end:** the same after the shift to Kickstarter's official rates.
- The ranking of campaigns is solid, the levels are rough. So the coach's slots are given by rank: the riskiest share of the cohort that the coach has capacity for (default 30%), ranked by the low end. *(Since 25.11 the verdict is separate from the slots: Coach first = low end below 11.5%, the cut at 30% capacity computed below; see the conclusions.)*
- To see what that means outside our sample, the 2025 test year is re-weighted so that each parent category has Kickstarter's official share of failures.

In [5]:
# ---- 3) chances and the verdict numbers
lo,hi=adv.chances(TE); TE['low']=lo; TE['high']=hi
s25=TE.groupby('parent_category').success.mean()
TE['w']=np.where(TE.success==0,TE.parent_category.map(lambda c:(s25[c]/(1-s25[c]))/(M.OFFICIAL_RATE[c]/(1-M.OFFICIAL_RATE[c]))),1.)   # re-weight to Kickstarter's official mix
print('re-weighted cohort: share failing',round((TE.w*(1-TE.success)).sum()/TE.w.sum(),3),'| sample: share failing',round(1-TE.success.mean(),3))
def reach(score,share,weights=None):
    w=np.ones(len(TE)) if weights is None else weights.values; o=np.argsort(score.values); cw=np.cumsum(w[o])/w.sum(); k=o[cw<=share]
    f=(1-TE.success.values)*w; return f[k].sum()/f.sum(), f[k].sum()/w[k].sum(), score.values[k].max()
rows=[]
for share in (.2,.3,.4,.5):
    a=reach(TE.high,share); b=reach(TE.low,share); c=reach(TE.low,share,TE.w)
    rows.append((share,round(a[0],3),round(b[0],3),round(b[1],3),round(c[0],3),round(c[1],3),round(c[2],3)))
V=pd.DataFrame(rows,columns=['coach capacity','sample: failures reached, rank by high end','sample: failures reached, rank by low end','sample: share of the group that fails','official mix: failures reached','official mix: share of the group that fails','low-end chance at the cut']); print(V.T.to_string()); V.to_csv(out+'/verdict_capacity.csv',index=False)
# within-category check: ranking by low end keeps the within-category AUC
print('AUC within category: high',round(M.auc_within(TE.success,TE.high,TE.cat_id),4),'low',round(M.auc_within(TE.success,TE.low,TE.cat_id),4),'| within parent: high',round(M.auc_within(TE.success,TE.high,TE.parent_category),4),'low',round(M.auc_within(TE.success,TE.low,TE.parent_category),4),'| overall: high',round(roc_auc_score(TE.success,TE.high),4),'low',round(roc_auc_score(TE.success,TE.low),4))
# who is in the riskiest 30% under each ranking, by parent
TE['risk_hi']=TE.high.rank(pct=True)<=.3; TE['risk_lo']=TE.low.rank(pct=True)<=.3
print(TE.groupby('parent_category').agg(n=('success','size'),in_riskiest_by_high=('risk_hi','mean'),in_riskiest_by_low=('risk_lo','mean')).round(2).T.to_string())
cut30=reach(TE.low,.3,TE.w)[2]; ART['verdict']={'reference_cut':float(cut30),'adjust_gain':5.0,'default_capacity':0.30}; adv.cut=ART['verdict']

re-weighted cohort: share failing 0.56 | sample: share failing 0.292
                                                 0      1      2      3
coach capacity                               0.200  0.300  0.400  0.500
sample: failures reached, rank by high end   0.576  0.759  0.872  0.938
sample: failures reached, rank by low end    0.572  0.747  0.865  0.932
sample: share of the group that fails        0.835  0.726  0.631  0.543
official mix: failures reached               0.348  0.511  0.659  0.785
official mix: share of the group that fails  0.976  0.954  0.922  0.880
low-end chance at the cut                    0.061  0.116  0.193  0.305


AUC within category: high 0.8587 low 0.8587 | within parent: high 0.8912 low 0.8912 | overall: high 0.9167 low 0.9104
parent_category          Art   Comics  Crafts  Dance   Design  Fashion  Film & Video    Food    Games  Journalism    Music  Photography  Publishing  Technology  Theater
n                    1763.00  2964.00  461.00  58.00  2416.00  1035.00       2593.00  974.00  6108.00      138.00  1182.00       412.00     3685.00     2499.00   245.00
in_riskiest_by_high     0.27     0.04    0.62   0.76     0.13     0.51          0.61    0.88     0.12        0.82     0.40         0.43        0.24        0.48     0.60
in_riskiest_by_low      0.25     0.07    0.55   0.43     0.22     0.57          0.45    0.72     0.16        0.65     0.29         0.38        0.33        0.47     0.21


## 6) What to change
For each campaign the module tries one change at a time on the levers a founder controls (goal in steps down to the minimum budget, duration, launch day, video, pre-launch page, more preparation time, title and blurb length) and keeps the best option per lever. Here on 4,000 campaigns of 2025, with the minimum budget assumed at half the goal.

In [6]:
# ---- 4) what to change: best single change for every 2025 campaign (minimum budget assumed = half the goal)
t0=time.time(); smp=TE.sample(4000,random_state=7); res=[]
for i,r in smp.iterrows():
    fx=adv.fixes(r); res.append((i,fx))
print('fixes s',round(time.time()-t0))
best=pd.DataFrame([{'i':i,'lever':fx[0]['lever'] if fx else 'none','gain':fx[0]['gain'] if fx else 0,'n':len(fx)} for i,fx in res]).set_index('i')
print('top fix by lever (share of campaigns):'); print(best.lever.value_counts(normalize=True).round(3).to_string())
print('gain of the best fix, points: median',best.gain.median(),'| p75',best.gain.quantile(.75),'| share >=5',round((best.gain>=5).mean(),3),'| >=10',round((best.gain>=10).mean(),3))
allfx=pd.DataFrame([dict(f,i=i) for i,fx in res for f in fx]); print('mean gain when the lever is offered:'); print(allfx.groupby('lever').gain.agg(['size','median','max']).round(1).to_string())
smp=smp.join(best); smp['risky']=smp.low<cut30
smp['verdict']=np.select([smp.risky,smp.gain>=5],['Coach first','Adjust'],'Launch')
print(smp.groupby('verdict').agg(n=('success','size'),share=('success',lambda s:len(s)/len(smp)),funded_in_sample=('success','mean'),low=('low','mean'),high=('high','mean')).round(3).to_string())
for g in (3,5,8,10): print('adjust threshold',g,'-> Adjust share among non-coach',round(((~smp.risky)&(smp.gain>=g)).sum()/(~smp.risky).sum(),3))
# direction check of the levers against the raw data (within category, 2023-24): does the model's advice agree with what we see?
print('monotonic check: share of campaigns where a 30% lower goal LOWERS the predicted chance:')
q=smp.copy(); q['log_goal_usd']-=np.log10(1/.7)*1; q['goal_ratio']+=np.log10(.7); q['log_goal_per_day']+=np.log10(.7); print(round((adv.chances(q)[1]<smp.high-1e-9).mean(),3))

fixes s 42
top fix by lever (share of campaigns):
lever
none               0.345
Pre-launch page    0.183
Preparation        0.151
Duration           0.110
Goal               0.100
Video              0.046
Launch day         0.032
Blurb              0.018
Title              0.014
Launch date        0.001
gain of the best fix, points: median 3.742554663851372 | p75 9.713253758414197 | share >=5 0.429 | >=10 0.243
mean gain when the lever is offered:
                 size  median   max
lever                              
Blurb             193     3.8  15.7
Duration         1132     3.7  29.2
Goal             1547     4.6  21.2
Launch date        24     2.9   8.9
Launch day        368     3.5  10.9
Pre-launch page  1137    10.8  23.4
Preparation      1174     6.7  43.1
Title             207     3.2  11.0
Video             482     4.2  16.8
                n  share  funded_in_sample    low   high
verdict                                                 
Adjust       1277  0.319             

**Note (26.11.2026):** this notebook was re-run on the raw data on 26.11 with the current code. Since 25.11 gains are measured on the middle estimate, the number the app shows, and the round-number flag of the goal keeps the plan's own value when a lower goal is tried. The output above matches `stage5_gain.py` and `docs/explanation_and_verdict.md`: 35% have no change worth 2 points, 43% have one worth 5 points or more. The verdict mix uses the cut computed on this data (11.6%); the app uses the fixed 11.5%, which gives 17 / 32 / 51% (`stage5_gain.py`).

**Interpretation:** for about a third of campaigns no single change adds 2 points (35% on the middle estimate). The largest associations are with a pre-launch page and more preparation time. Both stand for groundwork (building an audience before launch): opening a page the day before launch will not add these points.

## 7) Product types that do badly
The Market score mostly reflects what is being made. A very low score is a signal for the coach that the product type itself has rarely been funded on Kickstarter.

In [7]:
# ---- 5) product types that do badly: Market area
low=TE[TE.score_Market<=15]; print('Market score <=15: n',len(low),'funded in sample',round(low.success.mean(),3),'| others',round(TE[TE.score_Market>15].success.mean(),3))
print(low.groupby('parent_category').size().sort_values(ascending=False).head(6).to_dict()); print(low.subcategory.value_counts().head(10).to_dict())

Market score <=15: n 3447 funded in sample 0.214 | others 0.782
{'Technology': 968, 'Food': 582, 'Film & Video': 344, 'Publishing': 252, 'Fashion': 249, 'Art': 245}
{'Apps': 501, 'Web': 173, 'Software': 144, 'Restaurants': 144, 'Drinks': 125, 'Apparel': 120, 'Small Batch': 93, 'Toys': 77, 'Food Trucks': 73, 'Mobile Games': 66}


## 8) Explanation tree (class: decision tree)
A depth-3 tree on features a founder can read (no text score, no category codes). The app shows the campaign's path through it.

In [8]:
# explanation tree on features a founder can read (no text score, no category codes)
EX=[f for f in FM if f not in ('text_score','cat_id','parent_c','country_c')]
t3=DecisionTreeClassifier(criterion='entropy',max_depth=3,min_samples_leaf=500,random_state=42).fit(TR[EX].fillna(-1),TR.success)
p=t3.predict_proba(TE[EX].fillna(-1))[:,1]; print('readable depth-3 tree: 2025 AUC within category',round(M.auc_within(TE.success,p,TE.cat_id),3),'overall',round(roc_auc_score(TE.success,p),3))
print(export_text(t3,feature_names=EX,decimals=2,show_weights=True))

readable depth-3 tree: 2025 AUC within category 0.794 overall 0.853
|--- creator_prior_rate <= 0.66
|   |--- prelaunch_activated <= 0.50
|   |   |--- log_goal_usd <= 4.01
|   |   |   |--- weights: [4539.00, 4734.00] class: 1
|   |   |--- log_goal_usd >  4.01
|   |   |   |--- weights: [3465.00, 780.00] class: 0
|   |--- prelaunch_activated >  0.50
|   |   |--- log_goal_usd <= 4.12
|   |   |   |--- weights: [2391.00, 9331.00] class: 1
|   |   |--- log_goal_usd >  4.12
|   |   |   |--- weights: [1706.00, 1424.00] class: 0
|--- creator_prior_rate >  0.66
|   |--- creator_prior_funded <= 3.50
|   |   |--- prelaunch_activated <= 0.50
|   |   |   |--- weights: [300.00, 1943.00] class: 1
|   |   |--- prelaunch_activated >  0.50
|   |   |   |--- weights: [273.00, 6004.00] class: 1
|   |--- creator_prior_funded >  3.50
|   |   |--- creator_prior_rate <= 0.96
|   |   |   |--- weights: [53.00, 770.00] class: 1
|   |   |--- creator_prior_rate >  0.96
|   |   |   |--- weights: [75.00, 6116.00] class

## 9) Three example campaigns from 2025

In [9]:
# ---- 7) three example campaigns
for i in [smp[smp.verdict=='Coach first'].index[3],smp[smp.verdict=='Adjust'].index[3],smp[smp.verdict=='Launch'].index[3]]:
    r=TE.loc[i]; print('\n',r['name'][:70],'|',r.parent_category,'/',r.subcategory,'| goal $',round(r.goal_usd),'| outcome:',['failed','funded'][r.success])
    print('  chance range',round(r.low*100),'-',round(r.high*100),'% | areas',{a:int(r['score_'+a]) for a in A.AREAS},'| verdict',smp.verdict.get(i))
    for f in adv.fixes(r): print('   +',f['gain'],'pts:',f['change'])
EX=[f for f in FM if f not in ('text_score','cat_id','parent_c','country_c')]
from sklearn.tree import DecisionTreeClassifier
ART['tree_readable']=DecisionTreeClassifier(criterion='entropy',max_depth=3,min_samples_leaf=500,random_state=42).fit(TR[EX].fillna(-1),TR.success); ART['tree_features']=EX


 Help Fund an E-Commerce Platform for Minority Entrepreneurs | Technology / Web | goal $ 18000 | outcome: failed
  chance range 2 - 7 % | areas {'Ambition': 12, 'Timing': 92, 'Pitch': 89, 'Founder': 23, 'Presentation': 38, 'Market': 0} | verdict Coach first
   + 2.6233287014823548 pts: Open a pre-launch page before launch
   + 2.462406316333147 pts: Lower the goal by 50% (to about $9,000)

 Vinyl for "On The Prowl" by Wild Love Tigress | Music / Rock | goal $ 4000 | outcome: funded
  chance range 50 - 67 % | areas {'Ambition': 59, 'Timing': 59, 'Pitch': 45, 'Founder': 30, 'Presentation': 22, 'Market': 49} | verdict Adjust
   + 16.65701104907423 pts: Open a pre-launch page before launch
   + 6.024213074168927 pts: Take 30 more days to prepare before launch
   + 2.9178218661604083 pts: Lower the goal by 50% (to about $2,000)

 Arthropods • Insects Enamel Pins | Art / Illustration | goal $ 224 | outcome: funded
  chance range 80 - 92 % | areas {'Ambition': 88, 'Timing': 47, 'Pitch': 53, 

### Cohort check
Simulated cohorts of 40 campaigns drawn from 2025; the coach takes the 12 riskiest.

In [10]:
# cohort-mode verdict mix: simulated cohorts of 40 campaigns drawn from 2025
rng=np.random.default_rng(1); r=[]
for _ in range(500):
    c=TE.sample(40,random_state=int(rng.integers(1e9))); k=c.low.rank(method='first')<=12; r.append((c[k].success.mean(),(1-c[k].success).sum()/max((1-c.success).sum(),1),c[~k].success.mean()))
r=np.array(r); print('cohorts of 40, coach takes 12 (30%): funded share among the 12:',r[:,0].mean().round(3),'| share of the cohort\'s failures among the 12:',r[:,1].mean().round(3),'(5th-95th pct',np.percentile(r[:,1],5).round(2),np.percentile(r[:,1],95).round(2),') | funded share among the other 28:',r[:,2].mean().round(3))

cohorts of 40, coach takes 12 (30%): funded share among the 12: 0.292 | share of the cohort's failures among the 12: 0.744 (5th-95th pct 0.56 1.0 ) | funded share among the other 28: 0.887


## 10) Save

In [11]:
joblib.dump(ART,out+'/models_M.joblib'); TE[['id','low','high','success','parent_category','cat_id']+['score_'+a for a in A.AREAS]].to_parquet(out+'/reference_2025.parquet')

## Conclusions
1. Market and Founder explain more than half of the score; the levers a founder controls explain the rest.
2. Area scores separate campaigns well: weak areas go with clearly lower funded rates.
3. Verdict (since 25.11): Coach first (renamed **Rethink** on 27.11 in the app and the docs; this notebook keeps the old label until its next re-run) = lower value below 11.5%; Adjust = one change adds at least 5 points to the middle estimate; Launch = otherwise, which is not a promise. Coaching slots are separate: the riskiest teams by rank, up to the coach's capacity (default 30%).
4. In a cohort with Kickstarter's official mix, the riskiest 30% contain about half of all failures, and nearly all of them fail.
5. Advice is association, not cause.